<a href="https://colab.research.google.com/github/aly-elbana/Rag_Asses/blob/main/Rag_Project_NoteBook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-core langchain-community langchain-google-genai langchain-huggingface langchain-chroma gradio pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 

In [1]:
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from src.config import ensure_api_key
ensure_api_key()


In [ ]:
!ls

 sample_data  'Untitled Folder'  'Untitled Folder 1'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 79.4 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import time
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


BASE_DIR = Path.cwd()
timestamp = int(time.time())
DB_NAME = str(BASE_DIR / f"vector_db_{timestamp}")
KNOWLEDGE_BASE = str(BASE_DIR / "Untitled Folder")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
load_dotenv(override=True)

def fetch_documents():
    kb_path = Path(KNOWLEDGE_BASE)
    if not kb_path.exists():
        print(f"Error: {KNOWLEDGE_BASE} not found")
        return []

    pdf_paths = list(kb_path.rglob("*.pdf"))
    documents = []

    print(f"Found {len(pdf_paths)} PDFs. Processing")

    for pdf_path in pdf_paths:
        try:
            doc_type = pdf_path.parent.name
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
            for page in pages:
                page.metadata["doc_type"] = doc_type
                page.metadata["source_file"] = pdf_path.name
                documents.append(page)

            print(f"Loaded: {pdf_path.name}")
        except Exception as e:
            print(f"Skipping {pdf_path.name}: {e}")

    return documents

def create_chunks(documents):
    if not documents:
        return []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    return text_splitter.split_documents(documents)

def create_embeddings(chunks):
    if not chunks:
        print("Empty chunks list. Nothing to embed")
        return

    print(f"Generating embeddings for {len(chunks)} chunks")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )

    print(f"Ingestion complete. DB stored at: {DB_NAME}")
    return vectorstore

docs = fetch_documents()
if docs:
    chunks = create_chunks(docs)
    create_embeddings(chunks)
else:
    print("No documents found. Check knowledge base folder")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found 4 PDFs. Processing
Loaded: lec 4.pdf
Loaded: lec 3.pdf
Loaded: lec 6.pdf
Loaded: Networks _lec 5.pdf
Generating embeddings for 118 chunks
Ingestion complete. DB stored at: /content/vector_db_1772837988


In [ ]:
import os
from pathlib import Path
from google.colab import userdata
import gradio as gr
from pathlib import Path


from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
DB_NAME = str(Path("vector_db"))

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
RETRIEVAL_K = 10

vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant.
You are chatting with a user.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def fetch_context(question: str) -> list[Document]:
    return retriever.invoke(question)

def combined_question(question: str, history: list[dict] = []) -> str:
    prior = "\n".join(m["content"] for m in history if m["role"] == "user")
    return prior + "\n" + question

def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    combined = combined_question(question, history)
    docs = fetch_context(combined)

    context_text = "\n\n".join(doc.page_content for doc in docs)
    formatted_system_prompt = SYSTEM_PROMPT.format(context=context_text)

    messages = [SystemMessage(content=formatted_system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)
    return response.content, docs

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Test cell


chat_history = []

q1 = "what is the hub according to the docs"
ans1, docs1 = answer_question(q1, chat_history)
chat_history.append({"role": "user", "content": q1})
chat_history.append({"role": "assistant", "content": ans1})

q2 = "what was it's alternative according to lecture"
ans2, docs2 = answer_question(q2, chat_history)

print(ans2)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 58.281442165s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

In [ ]:
import gradio as gr
from pathlib import Path

def chat_wrapper(message, history):
    formatted_history = []

    for item in history:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            human, ai = item[0], item[1]
            if human:
                formatted_history.append({"role": "user", "content": str(human)})
            if ai:
                clean_ai = str(ai).split("\n\n---")[0]
                formatted_history.append({"role": "assistant", "content": clean_ai})

        elif isinstance(item, dict) and "role" in item and "content" in item:
            role = item["role"]
            content = str(item["content"])
            if role == "assistant" and "\n\n---" in content:
                content = content.split("\n\n---")[0]
            formatted_history.append({"role": role, "content": content})

        elif hasattr(item, "role") and hasattr(item, "content"):
            role = item.role
            content = str(item.content)
            if role == "assistant" and "\n\n---" in content:
                content = content.split("\n\n---")[0]
            formatted_history.append({"role": role, "content": content})

    answer, docs = answer_question(message, formatted_history)

    source_list = "\n\n---\n**Sources used:**"
    unique_sources = set()
    for doc in docs:
        source_name = doc.metadata.get('source', 'Unknown Document')
        unique_sources.add(Path(str(source_name)).name)

    for s in unique_sources:
        source_list += f"\n- {s}"

    return f"{answer}{source_list}"

demo = gr.ChatInterface(
    fn=chat_wrapper,
    title="Gemini RAG Assistant",
    description="Ask questions about your documents.",
)

if __name__ == "__main__":
    demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://316e09d94b6e7765b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://316e09d94b6e7765b0.gradio.live
